# Style-Aware Paraphraser — Demo

Companion notebook for *Attacks on Machine-Text Detectors Retain Stylistic Fingerprints* (ICML).

Pipeline (paper §4):

1. **Mistral-7B paraphrase**: paraphrase the input machine text P=5 times with `mistralai/Mistral-7B-Instruct-v0.3` (the model was trained on Mistral paraphrases, so this step is required even though it's a separate LLM).
2. **Pick a target author** from the author bank — `reference_text` (16 exemplars) + `paraphrase_reference_text` (5 Mistral paraphrases each) define the target style.
3. **Iterative refinement**: 3 rounds, 10 candidates per round, top-5 by SBERT against the original.

We load Mistral-7B and our DPO model **sequentially** so a single 80 GB A100 is enough.

> **Best on social-media-like text.** The model was trained on Reddit comments (32–128 tokens). It transfers to Amazon reviews and (more weakly) Blogs in the paper's evaluation. For long-form text, paraphrase paragraph-by-paragraph and concatenate — see Appendix L ("Chunk & Merge").


## 0. Install dependencies

In [ ]:
!pip install -r ../requirements.txt

## 1. Paraphrase the machine text with Mistral-7B

Load Mistral-7B-Instruct-v0.3, paraphrase the input P=5 times, then free
the model so the released DPO checkpoint has room on the GPU.


In [ ]:
import os, sys, gc
sys.path.insert(0, os.path.abspath('.'))
import torch
from vllm import LLM

from paraphrase_mistral import paraphrase_p5

user_machine_text = (
    "Climate change presents one of the most significant challenges of our time, "
    "requiring coordinated global action across sectors and nations. The latest "
    "IPCC report underscores the urgency of reducing greenhouse gas emissions."
)

mistral = LLM('mistralai/Mistral-7B-Instruct-v0.3', gpu_memory_utilization=0.85)
user_paraphrases = paraphrase_p5(user_machine_text, llm=mistral)
for p in user_paraphrases:
    print(' •', p[:200])

# Free Mistral so the released DPO model fits on the same GPU.
del mistral
gc.collect()
torch.cuda.empty_cache()


## 2. Pick a target author from the bank

The author bank contains 12 000 anonymous Reddit authors. Each row gives
us 16 exemplar comments + 5 Mistral-7B paraphrases of each — the
target-style context our model conditions on.


In [ ]:
from datasets import load_dataset
bank = load_dataset(
    'rrivera1849/style-aware-paraphraser-author-bank-reddit',
    split='train', streaming=True,
)
target = next(iter(bank))
print('target reference_text[0]:', target['reference_text'][0][:200])


## 3. Iterative refinement against the released DPO model

Load our model and run 3 iterations: each round generates 10 candidates
per source, top-5 by SBERT against the original machine text are kept
for the next round.


In [ ]:
from utils import MODEL_PATH
from iterative_refinement import iterative_refine
from embedding_utils import load_sbert_model

ours = LLM(MODEL_PATH, gpu_memory_utilization=0.85, max_model_len=15000)
sbert = load_sbert_model()
if torch.cuda.is_available():
    sbert = sbert.cuda()

iters = iterative_refine(
    initial_paraphrases=user_paraphrases,
    target_texts=target['reference_text'],
    target_paraphrases=target['paraphrase_reference_text'],
    llm=ours,
    sbert=sbert,
    original_text=user_machine_text,
    num_iters=3,
)
print('Final iteration outputs (top 5 by SBERT):')
for s in iters[-1]:
    print(' •', s[:300])


## 4. (Optional) Score the output

For an end-to-end check that the released pipeline still produces detectability comparable to the paper's headline numbers, see `release/validation/run_validation.py` — it scores LogRank (gpt2-xl) and StyleDetect (LUAR-MUD) on 500 Reddit rows and reproduces Figure 1's left panel.
